In [1]:
import os
import json
import time
import requests
from collections import defaultdict

import requests
import base64

In [2]:
ACCESS_TOKEN=""
SPOTIFY_SEARCH_URL = "https://api.spotify.com/v1/search"
SPOTIFY_ARTISTS_URL = "https://api.spotify.com/v1/artists"

HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

In [3]:
import os
import json

def extract_unique_artists(folder_path, output_file="unique_artists.json"):
    print("[Simbora] Extraindo artistas únicos...")
    artist_names = set()

    for filename in os.listdir(folder_path):
        if not filename.endswith(".json"):
            continue

        filepath = os.path.join(folder_path, filename)
        print(f"[Verificando] {filename}")

        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Normaliza para lista
        if isinstance(data, dict):
            records = [data]
        else:
            records = data

        if not records:
            continue

        # 🔥 Se já tiver campo de enriquecimento, pula arquivo
        first_record = records[0]
        if isinstance(first_record, dict) and (
            "artist_id" in first_record or
            "artist_genres" in first_record
        ):
            print(f"{filename} já está enriquecido.")
            continue

        print(f"[Lendo arquivinho] {filename}")

        for record in records:
            if not isinstance(record, dict):
                continue

            artist = record.get("master_metadata_album_artist_name")
            if artist:
                artist_names.add(artist)

    artist_list = sorted(list(artist_names))

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(artist_list, f, ensure_ascii=False, indent=2)

    print(f"[Vamo] {len(artist_list)} artistas salvos")

In [4]:
extract_unique_artists("../data/raw", output_file="../data/spotify-api/unique_artists.json")

[Simbora] Extraindo artistas únicos...
[Verificando] gigi10.json
[Lendo arquivinho] gigi10.json
[Verificando] gigi100.json
[Lendo arquivinho] gigi100.json
[Verificando] gigi123.json
[Lendo arquivinho] gigi123.json
[Verificando] Streaming_History_Audio_2019-2021_0.json
[Lendo arquivinho] Streaming_History_Audio_2019-2021_0.json
[Verificando] Streaming_History_Audio_2019-2024_0.json
[Lendo arquivinho] Streaming_History_Audio_2019-2024_0.json
[Verificando] Streaming_History_Audio_2020-2023_0Lana.json
[Lendo arquivinho] Streaming_History_Audio_2020-2023_0Lana.json
[Verificando] Streaming_History_Audio_2020-2024_0Fefo.json
[Lendo arquivinho] Streaming_History_Audio_2020-2024_0Fefo.json
[Verificando] Streaming_History_Audio_2021-2022_1.json
[Lendo arquivinho] Streaming_History_Audio_2021-2022_1.json
[Verificando] Streaming_History_Audio_2022-2023_4.json
[Lendo arquivinho] Streaming_History_Audio_2022-2023_4.json
[Verificando] Streaming_History_Audio_2022_2.json
[Lendo arquivinho] Streaming_H

In [5]:
import json
import time
import requests
import os

SPOTIFY_SEARCH_URL = "https://api.spotify.com/v1/search"

HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

def spotify_request(url, params=None):
    while True:
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=10)

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", 5))
                print(f"[TOMO NO PAPEIROKKKK] Esperando {retry_after}s...")
                time.sleep(retry_after + 2)
                continue

            response.raise_for_status()
            return response.json()

        except Exception as e:
            print(f"[TomonoPapeiro] {e} - retry em 5s")
            time.sleep(5)


def fetch_artist_ids(
    artists_file="../data/spotify-api/unique_artists.json",
    output_file="artist_id_map.json"
):
    with open(artists_file, "r", encoding="utf-8") as f:
        artist_names = json.load(f)

    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            artist_id_map = json.load(f)
    else:
        artist_id_map = {}

    total = len(artist_names)

    for i, name in enumerate(artist_names):
        if name in artist_id_map:
            continue  

        params = {"q": name, "type": "artist", "limit": 1}
        data = spotify_request(SPOTIFY_SEARCH_URL, params)

        items = data.get("artists", {}).get("items", [])
        artist_id_map[name] = items[0]["id"] if items else None

        if i % 10 == 0:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(artist_id_map, f, ensure_ascii=False, indent=2)

            print(f"[rodando] {i}/{total}")

        time.sleep(0.01)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(artist_id_map, f, ensure_ascii=False, indent=2)


In [6]:
fetch_artist_ids(output_file="../data/spotify-api/artist_id_map.json")

[rodando] 5650/6113
[rodando] 5660/6113
[rodando] 5670/6113
[rodando] 5680/6113
[rodando] 5690/6113
[rodando] 5700/6113
[rodando] 5710/6113
[rodando] 5720/6113
[rodando] 5730/6113
[rodando] 5740/6113
[rodando] 5750/6113
[rodando] 5760/6113
[rodando] 5770/6113
[rodando] 5780/6113
[rodando] 5790/6113
[rodando] 5800/6113
[rodando] 5810/6113
[rodando] 5820/6113
[rodando] 5830/6113
[rodando] 5840/6113
[rodando] 5850/6113
[rodando] 5860/6113
[rodando] 5870/6113
[rodando] 5880/6113
[rodando] 5890/6113
[rodando] 5900/6113
[rodando] 5910/6113
[rodando] 5920/6113
[rodando] 5930/6113
[rodando] 5940/6113
[rodando] 5950/6113
[rodando] 5960/6113
[rodando] 5970/6113
[rodando] 5980/6113
[rodando] 5990/6113
[rodando] 6000/6113
[rodando] 6010/6113
[rodando] 6020/6113
[rodando] 6030/6113
[rodando] 6040/6113
[rodando] 6050/6113
[rodando] 6060/6113
[rodando] 6070/6113
[rodando] 6080/6113
[rodando] 6090/6113
[rodando] 6100/6113
[rodando] 6110/6113


In [9]:
import json
import time
import requests
import os

SPOTIFY_ARTISTS_URL = "https://api.spotify.com/v1/artists"
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

def spotify_request(url, params=None):
    while True:
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=10)

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", 5))
                print(f"[Tomo no papeiroKKK] Esperando {retry_after}s...")
                time.sleep(retry_after + 2)
                continue

            response.raise_for_status()
            return response.json()

        except Exception as e:
            print(f" {e} - 5s")
            time.sleep(5)


def fetch_artist_genres(
    artist_id_file="../data/spotify-api/artist_id_map.json",
    output_file="artist_genres_map.json"
):
    with open(artist_id_file, "r", encoding="utf-8") as f:
        artist_id_map = json.load(f)

    ids = [v for v in artist_id_map.values() if v]

    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            id_to_genres = json.load(f)
    else:
        id_to_genres = {}

    for i in range(0, len(ids), 50):
        batch = ids[i:i+50]

        if all(aid in id_to_genres for aid in batch):
            continue

        params = {"ids": ",".join(batch)}
        data = spotify_request(SPOTIFY_ARTISTS_URL, params)

        for artist in data.get("artists", []):
            id_to_genres[artist["id"]] = artist.get("genres", [])

        if i % 200 == 0:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(id_to_genres, f, ensure_ascii=False, indent=2)

            print(f"[rodando] {i}/{len(ids)}")

        time.sleep(0.1)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(id_to_genres, f, ensure_ascii=False, indent=2)


In [10]:
fetch_artist_genres(output_file="../data/spotify-api/artists_genres_and_id.json")

[rodando] 0/6107
[rodando] 200/6107
[rodando] 400/6107
[rodando] 600/6107
[rodando] 800/6107
[rodando] 1000/6107
[rodando] 1200/6107
[rodando] 1400/6107
[rodando] 1600/6107
[rodando] 1800/6107
[rodando] 2000/6107
[rodando] 2200/6107
[rodando] 2400/6107
[rodando] 2600/6107
[rodando] 2800/6107
[rodando] 3000/6107
[rodando] 3200/6107
[rodando] 3400/6107
[rodando] 3600/6107
[rodando] 3800/6107
[rodando] 4000/6107
[rodando] 4200/6107
[rodando] 4400/6107
[rodando] 4600/6107
[rodando] 4800/6107
[rodando] 5000/6107
[rodando] 5200/6107
[rodando] 5400/6107
[rodando] 5600/6107
[rodando] 5800/6107
[rodando] 6000/6107


In [17]:
import os
import json

def enrich_records_in_place(
    folder_path,
    artist_id_file="../data/spotify-api/artist_id_map.json",
    genres_file="../data/spotify-api/artists_genres_and_id.json",
    overwrite=True
):

    with open(artist_id_file, "r", encoding="utf-8") as f:
        artist_id_map = json.load(f)

    with open(genres_file, "r", encoding="utf-8") as f:
        id_to_genres = json.load(f)

    print(f"[INFO] Artistas: {len(artist_id_map)}")
    print(f"[INFO] Generos: {len(id_to_genres)}")

    for filename in os.listdir(folder_path):
        if not filename.endswith(".json"):
            continue

        filepath = os.path.join(folder_path, filename)
        print(f"\n[Rodando] {filename}")

        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict):
            records = [data]
        else:
            records = data

        missing_id = 0

        for record in records:
            if not isinstance(record, dict):
                continue

            artist_name = record.get("master_metadata_album_artist_name")
            artist_id = artist_id_map.get(artist_name)

            if artist_id is None:
                missing_id += 1

            record["artist_id"] = artist_id
            record["artist_genres"] = (
                id_to_genres.get(artist_id, [])
                if artist_id else []
            )

        output_path = filepath if overwrite else os.path.join(
            folder_path,
            f"enriched_{filename}"
        )

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)

        print(f"[Funfo] Missing artist_id: {missing_id}")

    print("\n Funfo legal")

In [18]:
enrich_records_in_place("../data/raw")

[INFO] Artistas: 6113
[INFO] Generos: 5103

[Rodando] gigi10.json
[Funfo] Missing artist_id: 46

[Rodando] gigi100.json
[Funfo] Missing artist_id: 1

[Rodando] gigi123.json
[Funfo] Missing artist_id: 1

[Rodando] Streaming_History_Audio_2019-2021_0.json
[Funfo] Missing artist_id: 64

[Rodando] Streaming_History_Audio_2019-2024_0.json
[Funfo] Missing artist_id: 4

[Rodando] Streaming_History_Audio_2020-2023_0Lana.json
[Funfo] Missing artist_id: 39

[Rodando] Streaming_History_Audio_2020-2024_0Fefo.json
[Funfo] Missing artist_id: 134

[Rodando] Streaming_History_Audio_2021-2022_1.json
[Funfo] Missing artist_id: 134

[Rodando] Streaming_History_Audio_2022-2023_4.json
[Funfo] Missing artist_id: 10

[Rodando] Streaming_History_Audio_2022_2.json
[Funfo] Missing artist_id: 36

[Rodando] Streaming_History_Audio_2022_3.json
[Funfo] Missing artist_id: 13

[Rodando] Streaming_History_Audio_2023-2024_6.json
[Funfo] Missing artist_id: 2

[Rodando] Streaming_History_Audio_2023_5.json
[Funfo] Missing